# Data Source 06 · 경영·마케팅

| 출처 | 접근 | 검증 상태 |
|---|---|---|
| Wikimedia Pageviews REST API | 무키 (User-Agent 필요) | A |
| UCI Online Retail II (id 502, CC BY 4.0) | 수동 | A |
| UCI Bike Sharing (id 275, CC BY 4.0) | 수동 | A |
| Google Trends (`pytrends`) | 비공식 | C (권장 안 함) |

기본 경로는 Wikimedia 일별 조회수(관심도 대리변수)이다. 소매 거래·대여량은 UCI에서 내려받아 별도 셀로 읽는다.

In [ ]:
import io, json, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt

def fetch_bytes(url, timeout=30):
    """무키 공개 URL 요청. 실패 시 예외 -> 호출부에서 합성 대체."""
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def synthetic_series(n, freq, seed, trend=0.05, season=12, noise=1.0, start="2015-01-01"):
    """NOT real data: 방법 시연용 합성 시계열."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = 100 + trend * t + 3 * np.sin(2 * np.pi * t / season) + rng.normal(0, noise, n)
    return pd.Series(y, index=pd.date_range(start, periods=n, freq=freq), name="value")

def quality_report(s):
    s = s.sort_index()
    print("관측 수:", len(s), "| 기간:", s.index.min(), "~", s.index.max())
    print("결측:", int(s.isna().sum()), "| 중복 시점:", int(s.index.duplicated().sum()))
    f = pd.infer_freq(s.index) if len(s) > 2 else None
    print("추정 주기:", f)

In [ ]:
def parse_wikimedia_pageviews(obj):
    items = obj["items"]
    idx = pd.to_datetime([it["timestamp"][:8] for it in items], format="%Y%m%d")
    return pd.Series([it["views"] for it in items], index=idx, name="views")

In [ ]:
ARTICLE = "Time_series"
try:
    u = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/"
         f"{ARTICLE}/daily/20230101/20231231")
    y = parse_wikimedia_pageviews(json.loads(fetch_bytes(u)))
    DATA_KIND = "REAL: Wikimedia pageviews " + ARTICLE
except Exception as e:
    print("수집 실패 -> 합성 대체:", type(e).__name__, e)
    y = synthetic_series(365, "D", 2026061, trend=0.02, season=7, noise=2.0, start="2023-01-01")
    DATA_KIND = "SYNTHETIC (NOT real data)"
print("DATA_KIND:", DATA_KIND); quality_report(y); y.plot(title=DATA_KIND); plt.show()

## 점검 체크리스트
- 조회수는 봇/자동 접근 필터(`user` 에이전트) 정의에 따라 달라진다
- 개입(캠페인·이벤트) 날짜가 외생적인지 사전에 정의했는가 (Ch.7)
- Online Retail II: 취소/반품 행(송장번호 `C` 접두 등)과 결측 고객ID 처리는 데이터 설명 페이지로 확인
- 개인정보·라이선스 조건 확인. 원자료는 재배포하지 않는다